In [ ]:
#pip install scikit-surprise

# uncomment the above command if you don't have this package

  Using cached scikit_surprise-1.1.4.tar.gz (154 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-macosx_11_0_arm64.whl size=485368 sha256=277d2642be9c920e1f4666f549f3cfda27abacee839615732b65ad89c814de37
  Stored in directory: /Users/SasiVattikuti/Library/Caches/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise
Note: you may need to restart the kernel to use updated packages.


In [12]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
import pandas as pd

In [13]:
ratings = pd.read_csv("ml-32m/ratings.csv")

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2)

In [14]:
# SVD model
model = SVD()
model.fit(trainset)

# Evaluating performance
predictions = model.test(testset)
print("RMSE:", accuracy.rmse(predictions))

uid = str(1)    
iid = str(260)  
pred = model.predict(uid, iid)
print(f"Predicted rating of user {uid} for movie {iid}: {pred.est:.2f}")

RMSE: 0.7723
RMSE: 0.7722764414096
Predicted rating of user 1 for movie 260: 3.54


In [15]:
def get_top_n(predictions, n=5):
    from collections import defaultdict
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n


In [16]:
movies = pd.read_csv("ml-32m/movies.csv")
toy_story_id = movies[movies['title'].str.contains("Toy Story", case=False, na=False)]['movieId'].iloc[0]
toy_story_fans = ratings[(ratings['movieId'] == toy_story_id) & (ratings['rating'] >= 4)]
user_id = toy_story_fans['userId'].iloc[0]

# Recommending movies for this user
watched_movies = ratings[ratings['userId'] == user_id]['movieId'].tolist()
all_movie_ids = movies['movieId'].unique()
unwatched = [mid for mid in all_movie_ids if mid not in watched_movies]

#Predicting ratings for unseen movies
predictions = [model.predict(str(user_id), str(mid)) for mid in unwatched]
top_preds = sorted(predictions, key=lambda x: x.est, reverse=True)[:5]

# Mapping movieIds back to titles
top_movie_ids = [int(pred.iid) for pred in top_preds]
recommended_titles = movies[movies['movieId'].isin(top_movie_ids)][['movieId', 'title']]

print(f"\nTop 5 Recommendations for User {user_id} (Toy Story fan):")
print(recommended_titles.to_string(index=False))


Top 5 Recommendations for User 17 (Toy Story fan):
 movieId                              title
       2                     Jumanji (1995)
       3            Grumpier Old Men (1995)
       4           Waiting to Exhale (1995)
       5 Father of the Bride Part II (1995)
       7                     Sabrina (1995)
